## Attributions and Open Targets

We use gradient based explanations to attribute importance scores to the genes in a population of cells for a given phenotype (diseased cells)

In [ ]:
import requests
base_url = "https://www.ebi.ac.uk/ols/api/ontologies/efo/terms"

data = requests.get(url=base_url, params={"size":50, "page":0}).json()

In [36]:
data.keys()

dict_keys(['_embedded', '_links', 'page'])

In [58]:
import requests, tqdm
def get_disease_ids(size=1000):
    base_url = "https://www.ebi.ac.uk/ols/api/ontologies/efo/terms"
    num_pages = requests.get(url=base_url, params={"size": size, "page": 0}).json()['page']['totalPages']
    name_to_efo = {}
    for num in tqdm.tqdm(range(5)):
        data = requests.get(url=base_url, params={"size": size, "page": num}).json()['_embedded']['terms']
        name_to_efo.update({term["label"]: term["obo_id"].replace(':', '_') for term in data if term.get("obo_id", "").startswith(("EFO:", "MONDO:")) and term['is_root']==True})
    return name_to_efo
res = get_disease_ids(100)

100%|██████████| 5/5 [00:04<00:00,  1.02it/s]


In [69]:
def search_diseases(disease_names, ontology="efo"):
    base_url = "https://www.ebi.ac.uk/ols/api/search"
    results = {}
    for name in disease_names:
        response = requests.get(base_url, params={"q": name, "ontology": ontology, "exact": "true"})
        if response.ok:
            data = response.json()
            if data["response"]["docs"]:
                doc = data["response"]["docs"][0]
                results[name] = doc.get("obo_id", doc.get("short_form"))
    return results
search_diseases(['Parkinson disease', 'Alzheimer Disease', 'dementia'])

{'Parkinson disease': 'MONDO:0005180',
 'Alzheimer Disease': 'MONDO:0004975',
 'dementia': 'HP:0000726'}

In [ ]:
import scanpy as sc

ad = sc.read_h5ad('../../data/test_cxg.h5ad').obs

Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id', 'is_primary_data',
       'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id',
       'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue',
       'tissue_ontology_term_id', 'tissue_general',
       'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz',
       'raw_variance_nnz', 'n_measured_vars'],
      dtype='object')

In [ ]:
def get_associated_genes(disease_ontology_term_id, top=100):
    disease_ontology_term_id = disease_ontology_term_id.replace(':', '_')
    url = "https://api.platform.opentargets.org/api/v4/graphql"
    
    # disease associations query structure from a GraphQL API
    query = """
    query associatedTargets($diseaseId: String!) {
      disease(efoId: $diseaseId) {
        id
        name
        associatedTargets {
          count
          rows {
            target {
              id
              approvedSymbol
            }
            score
          }
        }
      }
    }
    """
    variables = {"diseaseId": disease_ontology_term_id}
    response = requests.post(url, json={"query": query, "variables": variables}).json()['data']['disease'] #post instead of .get() because its a REST API
    response_dict = {row['target']['approvedSymbol']: row['score'] for row in response['associatedTargets']['rows'][:top]}
    return response_dict

get_associated_genes("MONDO:0005180", 10)

{'LRRK2': 0.8739579473743275,
 'ATP13A2': 0.8531361870547747,
 'PRKN': 0.8505454789844968,
 'PINK1': 0.8490101765055016,
 'SNCA': 0.8467179432841184,
 'PARK7': 0.8071371449716728,
 'FBXO7': 0.8021364016972541,
 'DNAJC6': 0.8021270900445332,
 'GBA1': 0.7649315161562334,
 'VPS35': 0.7603253971350206}

In [10]:

ensembl_id_to_gene_name = {row[0]:row[1] for row in sc.read_h5ad("../../../data/test_cxg.h5ad").var[['feature_id', 'feature_name']].values}
import json
json.dump(ensembl_id_to_gene_name, open("../../data_utils/vocab/ensembl_to_gene.json", "w"))

In [11]:
import anndata as an
import scanpy as sc
import sys
sys.path.append('../../../')
from polygene.eval.metrics import prepare_cell, preprocess_logits_argmax
from polygene.data_utils.tokenization import GeneTokenizer
from tqdm import tqdm
import requests
import torch.nn.functional as F
import pandas as pd, numpy as np
from scipy.stats import fisher_exact
import torch
import json


class AttributionAnalysis():
    def __init__(self, model, tokenizer: GeneTokenizer , data = None, device="cuda:0", biotype_json="../../data_utils/vocab/gene_biotypes.json", ensembl_json="../../data_utils/vocab/ensembl_to_gene.json"):
        """
        Initialize Analyzer that computes feature attributions with different methods for a group of cells
        model: PyTorch Model
        tokenizer: tokenizer saved from pickle
        data: str or AnnData h5ad file with cells of interest
        """
        self.model, self.tok = model, tokenizer
        self.device = device
        model.to(device)
        self.data = sc.read_h5ad(data) if isinstance(data, str) else data
        self.list_of_ensembl_ids = list(self.tok.gene_type_id_map.keys())

        self.ensembl_id_to_gene_name = json.load(open(ensembl_json)) if ensembl_json is not None else None
        self.gene_biotypes = json.load(open(biotype_json)) if biotype_json is not None else None
    
    def __call__(self, method='integrated_gradients', data=None):
        if data is None and self.data is None: return print("Missing data")

    def gradients(self, phenotype="disease", reduction_func = None, only_protein_encoding=True):
        # reduction function needs to take a (S, D) matrix and return an (S,) vector. 
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        
        cell_attributions = []
        for cell in tqdm(self.data):
            x = prepare_cell(cell, self.tok)
            phenotype_value = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]
            x['inputs_embeds'] = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach().requires_grad_(True)

            y = F.softmax(self.model(**{k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k != 'str_labels'}).logits, dim=-1) # (B, S, D)
            Ly = y[0, 1+self.tok.phenotypic_types.index(phenotype), self.tok.flattened_tokens.index(phenotype_value)]
            self.model.zero_grad()
            Ly.backward()

            attributions_per_gene = reduction_func(x['inputs_embeds'].grad.detach().cpu().numpy()[self.tok.gene_token_type_offset:])
            gene_ensembl_ids = [self.list_of_ensembl_ids[gene - self.tok.gene_token_type_offset]
                                 for gene in x['token_type_ids'].detach().cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append( {k:v for k,v in zip(gene_ensembl_ids, attributions_per_gene)})
        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        
        if only_protein_encoding and self.gene_biotypes is not None: # This can take an extra 2-3 minutes, perhaps save protein encoding information in a json
            ensembl_ids =  self.cell_attributions_df.columns.tolist()
            mask = [eid for eid in ensembl_ids if self.gene_biotypes.get(eid, "") == "protein_coding"]
            print(f"Computed Attributions for {len(ensembl_ids)} genes and kept {len(mask)} protein encoding genes")
            self.cell_attributions_df = self.cell_attributions_df.loc[:, np.array(mask)]

        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df

    def integrated_gradients(self, phenotype="disease", reduction_func=None, steps=10, only_protein_encoding=True):
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        cell_attributions = []
        for cell in tqdm(self.data):
            x = prepare_cell(cell, self.tok); phv = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]
            inp = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach()
            base = torch.zeros_like(inp).to(self.device); diff = inp-base; grads = torch.zeros_like(inp)
            for a in torch.linspace(0,1,steps):
                scaled = (base+a*diff).detach().requires_grad_(True); feed={k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k!='str_labels'}; feed['inputs_embeds']=scaled.unsqueeze(0)
                y = F.softmax(self.model(**feed).logits,dim=-1); Ly=y[0,1+self.tok.phenotypic_types.index(phenotype),self.tok.flattened_tokens.index(phv)]; self.model.zero_grad(); Ly.backward(retain_graph=True); grads+=scaled.grad.detach()
            ig=(diff*grads/steps).detach().cpu().numpy(); vals=reduction_func(ig[self.tok.gene_token_type_offset:]); ids=[self.list_of_ensembl_ids[i-self.tok.gene_token_type_offset] for i in x['token_type_ids'].cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append(dict(zip(ids,vals)))
        self.cell_attributions_df=pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding and self.gene_biotypes is not None: # This can take an extra 2-3 minutes, perhaps save protein encoding information in a json
            ensembl_ids =  self.cell_attributions_df.columns.tolist()
            mask = [eid for eid in ensembl_ids if self.gene_biotypes.get(eid, "") == "protein_coding"]
            print(f"Computed Attributions for {len(ensembl_ids)} genes and kept {len(mask)} protein encoding genes")
            self.cell_attributions_df = self.cell_attributions_df.loc[:, np.array(mask)]

        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df
    

    def deep_lift(self, phenotype="disease", reduction_func=None, only_protein_encoding=True):
        from captum.attr import DeepLift
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        cell_attributions=[]
        for cell in tqdm(self.data):
            x = prepare_cell(cell, self.tok); phv = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]
            feed = {k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k!='str_labels'}
            feed['inputs_embeds'] = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach().requires_grad_(True).unsqueeze(0)
            base = torch.zeros_like(feed['inputs_embeds']).to(self.device)

            class ForwardWrapper(torch.nn.Module):
                def __init__(sself, model): super().__init__(); sself.model=model
                def forward(sself, inputs_embeds, attention_mask): return sself.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits
            dl = DeepLift(ForwardWrapper(self.model))

            target = (1+self.tok.phenotypic_types.index(phenotype),self.tok.flattened_tokens.index(phv))

            out = dl.attribute(inputs=feed['inputs_embeds'], baselines=base, additional_forward_args=(feed["attention_mask"],), target=target)
            
            vals = reduction_func(out[0].detach().cpu().numpy()[self.tok.gene_token_type_offset:])
            ids = [self.list_of_ensembl_ids[i-self.tok.gene_token_type_offset] for i in x['token_type_ids'].cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append(dict(zip(ids, vals)))
        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding and self.gene_biotypes is not None: # This can take an extra 2-3 minutes, perhaps save protein encoding information in a json
            ensembl_ids =  self.cell_attributions_df.columns.tolist()
            mask = [eid for eid in ensembl_ids if self.gene_biotypes.get(eid, "") == "protein_coding"]
            print(f"Computed Attributions for {len(ensembl_ids)} genes and kept {len(mask)} protein encoding genes")
            self.cell_attributions_df = self.cell_attributions_df.loc[:, np.array(mask)]

        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df
        
    @staticmethod
    def get_associated_genes(disease_ontology_term_id, top=100):
        disease_ontology_term_id = disease_ontology_term_id.replace(':', '_')
        url = "https://api.platform.opentargets.org/api/v4/graphql"
        
        # disease associations query structure from a GraphQL API
        query = """
            query associatedTargets($diseaseId: String!, $size: Int!) {
                disease(efoId: $diseaseId) {
                    id
                    name
                    associatedTargets(page: { index: 0, size: $size }) {
                    count
                    rows { target { id approvedSymbol } score }
                    }
                }
            }
        """
        variables = {"diseaseId": disease_ontology_term_id, "size": top}
        response = requests.post(url, json={"query": query, "variables": variables}).json()['data']['disease'] #post instead of .get() because its a REST API
        response_dict = {row['target']['approvedSymbol']: row['score'] for row in response['associatedTargets']['rows'][:top]}
        return response_dict
    
    def validate_attributions(self, k=100, method_top_attr = "above_mean", phenotype_obs_key="disease", phenotype_obs_value="normal", baseline=None, 
                              start_with_top_X_genes=None):
        # assuming all cells are part of same phenotype
        ontology_id = self.data.obs[self.data.obs[phenotype_obs_key] == phenotype_obs_value]['disease_ontology_term_id'].tolist()[0]
        attributed_genes = self.cell_attributions_df.loc[(self.data.obs[phenotype_obs_key] == phenotype_obs_value).tolist(),:].sum(axis=0) if baseline is None else baseline
        if start_with_top_X_genes is not None: attributed_genes = attributed_genes.sort_values(ascending=False)[:start_with_top_X_genes]
        
        opentarget_dict = self.get_associated_genes(ontology_id, k)
        opentarget_genes = set(opentarget_dict.keys())

        if method_top_attr == "above_mean":
            print("attribution mean:", round(attributed_genes.mean(), 5))
            topk_attributed = set(attributed_genes[attributed_genes > attributed_genes.mean()].index.tolist())
        elif method_top_attr == "Q3":
            topk_attributed = set(attributed_genes[attributed_genes > attributed_genes.quantile(q=0.75)].index.tolist())
        else: 
            topk_attributed = set(attributed_genes.sort_values(ascending=False)[:k].index.tolist())

        overlap_genes = opentarget_genes & topk_attributed
        overlap = len(overlap_genes)
        TP, FP, FN = overlap, len(topk_attributed) - overlap, len(opentarget_genes) - overlap
        TN = len(attributed_genes) - (TP+FP+FN)
        recall, precision = TP / (TP + FN), TP / (TP + FP)
        odds, pval = fisher_exact([[TP,FP],[FN,TN]], alternative="greater")
        print(f"{overlap} common genes from {k} open target genes and {len(topk_attributed)} attributed genes")
        print(f"\nRecall: {recall:.3f}, Precision/Novelty: {precision:.3f}, Fisher's Exact p: {pval:.5f}, " 
              f"Overlap Strength: {sum([opentarget_dict.get(gene) for gene in overlap_genes]):.2f}, Overlap Genes: {sorted(overlap_genes)}")
        print(f"Candidate novel genes:{attributed_genes.drop(list(overlap_genes)).sort_values(ascending=False).index.tolist()[:3]}")
        return recall, precision, pval, sorted(overlap_genes)

    def baselines(self, phenotype_obs_key='disease', case_label=None, control_label=None, k=50, method_top_attr="Q3", start_with_X=None):
        from scipy.stats import ttest_ind
        from sklearn.metrics import mutual_info_score
        X = self.data.X.toarray()

        # fair comparison
        print('Lower bin edge:', self.tok.config.bin_edges[0])
        X[X < self.tok.config.bin_edges[0]] = 0
        nonzero_mask = X.sum(axis=0) > 0
        X = X[:, nonzero_mask]
        var_names = self.data.var_names[nonzero_mask]

        y = self.data.obs[phenotype_obs_key].to_numpy()
        mask_case, mask_ctrl = (y == case_label), (y == control_label)
        Xc, Xn = X[mask_case], X[mask_ctrl]
        t_stat, _ = ttest_ind(Xc, Xn, axis=0, equal_var=False, nan_policy='omit')

        y_bin = np.where(y == case_label, 1, 0)
        mi_scores = []
        for j in range(X.shape[1]):
            xj = np.asarray(X[:, j]).ravel()
            bins = np.quantile(xj, np.linspace(0, 1, 6))
            xj_disc = np.digitize(xj, bins, right=True)
            mi_scores.append(mutual_info_score(xj_disc, y_bin))

        print("GWAS baseline:")
        baseline = pd.Series(t_stat, index=[self.ensembl_id_to_gene_name.get(k,k) for k in var_names])
        self.validate_attributions(phenotype_obs_key=phenotype_obs_key, phenotype_obs_value=case_label, baseline=baseline, method_top_attr=method_top_attr, k=k, start_with_top_X_genes=start_with_X)
        print("Mutual Information baseline:")
        baseline = pd.Series(mi_scores, index=[self.ensembl_id_to_gene_name.get(k,k) for k in var_names])
        self.validate_attributions(phenotype_obs_key=phenotype_obs_key, phenotype_obs_value=case_label, baseline=baseline, method_top_attr=method_top_attr, k=k, start_with_top_X_genes=start_with_X)

# Baselines: average expression ranked, GWAS significance

In [26]:
from polygene.model.model import load_trained_model
import scanpy as sc

model_path = "../../../runs/gesam_polygene_run_4/" #'/media/lleger/LaCie/POLYGENE/'
model, tokenizer = load_trained_model(model_path, checkpoint_n=-1)
cells = sc.read_h5ad("/media/lleger/LaCie/dump/disease-vector/analogical/neurological_cells.h5ad")
cells = cells[cells.obs['disease'].isin(['Alzheimer disease'])][:200]
analyzer = AttributionAnalysis(model, tokenizer, data=cells)#, biotype_json="../data_utils/vocab/gene_biotypes.json")
df = analyzer.gradients(only_protein_encoding=True)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease")#, start_with_top_X_genes=int(1e4))
# to get 44 drop the start_with_X

100%|██████████| 200/200 [00:03<00:00, 51.81it/s]


Computed Attributions for 16536 genes and kept 13809 protein encoding genes
21 common genes from 50 open target genes and 3452 attributed genes

Recall: 0.420, Precision/Novelty: 0.006, Fisher's Exact p: 0.00617, Overlap Strength: 12.78, Overlap Genes: ['ABCA7', 'ADAM10', 'APOE', 'APP', 'AXL', 'CDK5R1', 'CDKN1A', 'CLU', 'FOXO3', 'GRIN1', 'GRIN2A', 'GRIN2B', 'GRN', 'HMGCR', 'NCK2', 'NDUFS2', 'NDUFS5', 'PLCG2', 'PSAP', 'SORL1', 'TREM2']
Candidate novel genes:['SLC26A3', 'RPS18', 'RASGEF1B']


In [27]:
# BASELINES 
# Reload cells because GWAS/MI needs control labels
cells = sc.read_h5ad("/media/lleger/LaCie/dump/disease-vector/analogical/neurological_cells.h5ad")
cells = cells[cells.obs['disease'].isin(['Alzheimer disease', 'normal'])][:200]
analyzer = AttributionAnalysis(model, tokenizer, data=cells)
r = analyzer.baselines(phenotype_obs_key="disease", case_label="Alzheimer disease", control_label="normal", method_top_attr="Q3", k=50)#, start_with_X=int(1e4))

Lower bin edge: 0.5
GWAS baseline:
18 common genes from 50 open target genes and 5279 attributed genes

Recall: 0.360, Precision/Novelty: 0.003, Fisher's Exact p: 0.05489, Overlap Strength: 10.75, Overlap Genes: ['ABCA7', 'ACE', 'ACHE', 'AURKB', 'CDK5', 'CDK5R1', 'CLU', 'GRIN1', 'GRIN2A', 'GRIN2B', 'HMGCR', 'LSM6', 'NDUFA8', 'NDUFAB1', 'NDUFS2', 'NDUFS5', 'PSAP', 'PSEN2']
Candidate novel genes:['LINGO1', 'IQCJ-SCHIP1', 'RASGEF1B']
Mutual Information baseline:
15 common genes from 50 open target genes and 5276 attributed genes

Recall: 0.300, Precision/Novelty: 0.003, Fisher's Exact p: 0.25088, Overlap Strength: 9.45, Overlap Genes: ['ADAM10', 'APP', 'CDC25B', 'CDK5R1', 'CLU', 'FOXO3', 'GRIN1', 'GRIN3A', 'GSK3B', 'HMGCR', 'NDUFS5', 'PLCG2', 'PSAP', 'PSEN1', 'SORL1']
Candidate novel genes:['SNHG14', 'MIR100HG', 'LINGO1']


In [28]:
# Summing the tokens (previous method)
df = analyzer.gradients(only_protein_encoding=True, reduction_func=lambda m: np.sum(m, axis=1))
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease", start_with_top_X_genes=int(1e4))

100%|██████████| 200/200 [00:05<00:00, 35.89it/s]


Computed Attributions for 18295 genes and kept 13929 protein encoding genes
9 common genes from 50 open target genes and 2500 attributed genes

Recall: 0.180, Precision/Novelty: 0.004, Fisher's Exact p: 0.90895, Overlap Strength: 5.67, Overlap Genes: ['ADAM10', 'APOE', 'APP', 'BCHE', 'CDKN2A', 'CLU', 'FOXO3', 'NCK2', 'PLCG2']
Candidate novel genes:['CHPT1', 'PCDH9', 'HYDIN']


#### advantages: we get attributions per cell/endotype properly, we outperform GWAS on enrichment analysis, we get sparser attributions

In [30]:
# INTEGRATED GRADIENTS

model_path = "../../../runs/gesam_polygene_run_4/" #'/media/lleger/LaCie/POLYGENE/'
model, tokenizer = load_trained_model(model_path, checkpoint_n=-1)
cells = sc.read_h5ad("/media/lleger/LaCie/dump/disease-vector/analogical/neurological_cells.h5ad")
cells = cells[cells.obs['disease'].isin(['Alzheimer disease'])][:100]
analyzer = AttributionAnalysis(model, tokenizer, data=cells,)
df = analyzer.integrated_gradients(only_protein_encoding=True, steps=5)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease",)# start_with_top_X_genes=int(1e4))

100%|██████████| 100/100 [00:09<00:00, 10.82it/s]


Computed Attributions for 15264 genes and kept 13011 protein encoding genes
18 common genes from 50 open target genes and 3253 attributed genes

Recall: 0.360, Precision/Novelty: 0.006, Fisher's Exact p: 0.05481, Overlap Strength: 11.14, Overlap Genes: ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CDK5R1', 'CLU', 'FOXO3', 'GRIN1', 'GRIN2A', 'GRIN2B', 'GRN', 'HMGCR', 'NDUFS2', 'NDUFS5', 'PLCG2', 'PSAP', 'SORL1', 'TREM2']
Candidate novel genes:['RP11-701H24.9', 'SLC26A3', 'RASGEF1B']


In [33]:
# DEEPLIFT

analyzer = AttributionAnalysis(model, tokenizer, data=cells,)
df = analyzer.deep_lift(only_protein_encoding=True)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease",)

  0%|          | 0/100 [00:00<?, ?it/s]

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/captum/attr/_core/deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  warnings.warn(
100%|██████████| 100/100 [00:03<00:00, 29.26it/s]


Computed Attributions for 15264 genes and kept 13011 protein encoding genes
20 common genes from 50 open target genes and 3253 attributed genes

Recall: 0.400, Precision/Novelty: 0.006, Fisher's Exact p: 0.01377, Overlap Strength: 12.37, Overlap Genes: ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CDK5', 'CDK5R1', 'CLU', 'FOXO3', 'GRIN1', 'GRIN2A', 'GRIN2B', 'GRN', 'HMGCR', 'NCK2', 'NDUFS2', 'NDUFS5', 'PLCG2', 'PSAP', 'SORL1', 'TREM2']
Candidate novel genes:['RASGEF1B', 'SLC26A3', 'RP11-701H24.9']
